# Interpreting confidence intervals
Repeated sampling shows what "95% confidence" means; two common misreadings are checked numerically; the margin of error is computed as the confidence level, the population spread and the sample size change.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy import stats

## 1. Coverage: how many intervals contain the population mean?
Population N(50, 15^2), samples of 50, 95% z-intervals with sigma known.

In [ ]:
def intervals(n=50, sims=100, level=0.95, mu=50, sigma=15, seed=42):
    # one row per sample; returns the lower and upper limits of each interval
    rng = np.random.default_rng(seed)
    means = rng.normal(mu, sigma, (sims, n)).mean(axis=1)
    margin = stats.norm.ppf((1 + level) / 2) * sigma / np.sqrt(n)
    return means - margin, means + margin

for sims in [100, 1000, 100_000]:
    low, high = intervals(sims=sims)
    print(f"{sims:6d} intervals: {np.mean((low <= 50) & (50 <= high)):.2%} contain mu = 50")

In [ ]:
def coverage_plot(n=50, sims=100, level=0.95):
    # call with other values to explore, e.g. coverage_plot(n=10, sims=300, level=0.8)
    low, high = intervals(n, sims, level)
    hit = (low <= 50) & (50 <= high)
    fig = go.Figure()
    for i in range(sims):
        fig.add_scatter(x=[i, i], y=[low[i], high[i]], mode="lines", showlegend=False,
                        line=dict(color="#4C78A8" if hit[i] else "#F58518"))
    fig.add_hline(y=50, line_color="red")
    fig.update_layout(template="simple_white", title=f"{hit.sum()} of {sims} intervals contain the mean ({hit.mean():.0%})")
    fig.show()

coverage_plot()

## 2. Misreading 1: "95% of future sample means fall inside this interval"
Fix one interval from one sample, then draw many new samples and count how many of their means land inside.

In [ ]:
rng = np.random.default_rng(1)
se = 15 / np.sqrt(50)
first = rng.normal(50, 15, 50).mean()
low, high = first - 1.96 * se, first + 1.96 * se
new_means = rng.normal(50, 15, (100_000, 50)).mean(axis=1)
print(f"one interval {low:.2f} to {high:.2f}: {np.mean((low <= new_means) & (new_means <= high)):.1%} of new means inside")
# averaged over many first samples, the share is P(|Z| < 1.96 / sqrt(2))
print("average share over all first samples:", round(2 * stats.norm.cdf(1.96 / np.sqrt(2)) - 1, 3))

## 3. Misreading 2: "95% of the values lie inside the interval"

In [ ]:
ages = rng.normal(50, 15, 100_000)       # individual values from the population
print(f"share of individual values inside {low:.2f} to {high:.2f}: {np.mean((low <= ages) & (ages <= high)):.1%}")

## 4. The three factors of the margin of error, E = z * sigma / sqrt(n)

In [ ]:
def margin(level=0.95, sigma=15, n=50):
    return stats.norm.ppf((1 + level) / 2) * sigma / np.sqrt(n)

for level in [0.5, 0.8, 0.9, 0.95, 0.99, 0.999]:
    print(f"level {level:.1%}: E = {margin(level=level):.2f}")
for sigma in [5, 15, 30]:
    print(f"sigma {sigma}: E = {margin(sigma=sigma):.2f}")
for n in [10, 30, 50, 120, 500, 1000]:
    print(f"n {n}: E = {margin(n=n):.2f}")

## 5. Sample size needed for a target margin of error

In [ ]:
for target in [4, 2, 1]:
    n_needed = (1.96 * 15 / target) ** 2
    print(f"E = {target}: n = {n_needed:.1f}, round up to {int(np.ceil(n_needed))}")